# 04 · BreathNet explained: how 6,251 parameters estimate breathing rate

BreathNet comes within 0.07–0.08 breaths/min (per-patient MAE) of a 400k-parameter CNN on both PPG and ECG,
with 64 times fewer parameters. The gap is small but consistent: the CNN is better for about two thirds of the
test patients (Wilcoxon p = 0.03 PPG, 0.01 ECG). This notebook opens BreathNet up: what each part learned, which parts the accuracy depends on, and what the
model looks at in individual windows. Sections 1–7 open up the fold-0 model on its held-out
patients; section 8 checks the same readings in every cross-validation fold. Model internals are computed by `python -m scripts.breathnet_analysis` (needs torch) and only plotted here.

**What we find**

1. **Frequency analysis is built in, not learned.** 57 resonators sit at fixed rates and the output bin *is*
   the rate, so there is no feature-to-rate head to learn. Only the resonator widths are trained (57 numbers).
2. **One pathway is turned into a background detector.** All three pathways share a slow 1/f background that
   pulls estimates low. A 267-parameter layer learned to *invert* one pathway (PPG: AM, ECG: BW) so that, when
   the pathways are summed, the background cancels. The model learned for itself the correction we hand-built
   for the signal-processing methods. Removing that layer costs +11 to +12 bpm.
3. **The three pathways are a product of experts.** Each alone is biased (PPG BW −7, FM −6, AM +6 bpm); summed
   in the log domain the biases cancel to ~1 bpm.
4. **Harmonic summation is the cheapest win:** 9 weights, worth 1.1–1.5 bpm. They include a *negative* weight
   at half the rate on every pathway, an octave-error veto.
5. **Attention and the gate are refinements**, worth 0.1–0.3 bpm, although they hold 94 % of the parameters.

## The architecture

<svg viewBox="0 0 1060 320" width="100%" style="max-width:1060px;font-family:system-ui,sans-serif" xmlns="http://www.w3.org/2000/svg">
<defs><marker id="a" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#898781"/></marker></defs>
<style>.b{fill:#fcfcfb;stroke:#c3c2b7;stroke-width:1.2}.t{font-size:13.5px;font-weight:600;fill:#0b0b0b}.s{font-size:11.5px;fill:#52514e}.p{font-size:12px;font-weight:600;fill:#2a78d6}.l{stroke:#898781;stroke-width:1.4;fill:none;marker-end:url(#a)}</style>
<text class="s" x="6" y="20" style="font-size:12.5px">BreathNet · 6,251 parameters in total (the CNN baseline has 401,769)</text>
<rect class="b" x="6" y="36" width="150" height="230" rx="10"/><text class="t" x="18" y="60">Input</text>
<text class="s" x="18" y="80">60 s of ECG or PPG</text><text class="s" x="18" y="96">→ beats → 3 pathways</text>
<rect x="18" y="110" width="126" height="26" rx="6" fill="#2a78d6" opacity=".16"/><text class="s" x="26" y="127">BW · baseline</text>
<rect x="18" y="142" width="126" height="26" rx="6" fill="#eb6834" opacity=".16"/><text class="s" x="26" y="159">AM · amplitude</text>
<rect x="18" y="174" width="126" height="26" rx="6" fill="#1baf7a" opacity=".16"/><text class="s" x="26" y="191">FM · beat intervals</text>
<text class="s" x="18" y="248">3 × 240 samples, 4 Hz</text>
<path class="l" d="M156,150 L178,150"/>
<rect class="b" x="180" y="36" width="150" height="230" rx="10"/><text class="t" x="192" y="60">Resonator bank</text>
<text class="s" x="192" y="80">57 damped oscillators,</text><text class="s" x="192" y="96">one per rate 4–60</text>
<text class="s" x="192" y="112">breaths/min.</text><text class="s" x="192" y="132">Frequency fixed,</text>
<text class="s" x="192" y="148">width learned.</text><text class="s" x="192" y="168">Output: energy map,</text><text class="s" x="192" y="184">rate × time</text>
<text class="p" x="192" y="248">57 parameters</text>
<path class="l" d="M330,150 L352,150"/>
<rect class="b" x="354" y="36" width="150" height="230" rx="10"/><text class="t" x="366" y="60">Harmonic sum</text>
<text class="s" x="366" y="80">Adds energy at 2× and</text><text class="s" x="366" y="96">3× the rate (breaths</text>
<text class="s" x="366" y="112">are not sinusoids).</text><text class="s" x="366" y="132">Subtracts ½× to veto</text><text class="s" x="366" y="148">octave errors.</text>
<text class="p" x="366" y="248">9 parameters</text>
<path class="l" d="M504,150 L526,150"/>
<rect class="b" x="528" y="36" width="150" height="230" rx="10"/><text class="t" x="540" y="60">Refine</text>
<text class="s" x="540" y="80">Tiny conv along the</text><text class="s" x="540" y="96">rate axis. Learned to</text>
<text class="s" x="540" y="112">invert one pathway</text><text class="s" x="540" y="128">into a 1/f background</text><text class="s" x="540" y="144">estimate.</text>
<text class="p" x="540" y="248">267 parameters</text>
<path class="l" d="M678,150 L700,150"/>
<rect class="b" x="702" y="36" width="150" height="230" rx="10"/><text class="t" x="714" y="60">Frame attention</text>
<text class="s" x="714" y="80">Scores each of 60</text><text class="s" x="714" y="96">one-second frames and</text>
<text class="s" x="714" y="112">pools the evidence over</text><text class="s" x="714" y="128">the steady part.</text>
<text class="p" x="714" y="248">5,667 parameters</text>
<path class="l" d="M852,150 L874,150"/>
<rect class="b" x="876" y="36" width="178" height="230" rx="10"/><text class="t" x="888" y="60">Gate + prior → rate</text>
<text class="s" x="888" y="80">Reliability weight per</text><text class="s" x="888" y="96">pathway (learned smart</text>
<text class="s" x="888" y="112">fusion). Evidence summed</text><text class="s" x="888" y="128">in the log domain, plus a</text>
<text class="s" x="888" y="144">rate prior. Output bin k</text><text class="s" x="888" y="160"><tspan font-style="italic">is</tspan> rate k: no head.</text>
<text class="p" x="888" y="248">193 + 57 + 1 parameters</text>
<text class="s" x="6" y="298" style="font-size:12.5px">Frequency analysis is built into the structure (fixed-frequency resonators whose index is the answer),</text>
<text class="s" x="6" y="314" style="font-size:12.5px">so the network only learns how to weigh evidence.</text>
</svg>

In [1]:
import sys, os, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "estimate_rr").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, Markdown, display

BLUE, ORANGE, AQUA, INK, INK2, GRAY, VIOLET = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#52514e", "#c3c2b7", "#6250d6"
PCOL = {"BW": BLUE, "AM": ORANGE, "FM": AQUA, "fused": INK}
SIGCOL = {"PPG": "#e87ba4", "ECG": VIOLET}   # signals get their own hues; blue/orange/aqua mean BW/AM/FM
_js = False
def show(fig, height=None):
    """Render with plotly.js embedded once, so the exported page works offline."""
    global _js
    fig.update_layout(template="simple_white", height=height, margin=dict(l=60, r=24, t=70, b=125),
                      font=dict(family="system-ui, sans-serif", size=12, color=INK2), title_font=dict(size=15, color=INK),
                      legend=dict(orientation="h", yanchor="top", y=-0.24, x=0, xanchor="left"))
    display(HTML(fig.to_html(include_plotlyjs=not _js, full_html=False))); _js = True

D = Path("runs/breathnet_f0")
art = {s.upper(): np.load(D / f"{s}.npz") for s in ("ppg", "ecg") if (D / f"{s}.npz").exists()}
abl = {s.upper(): pd.read_csv(D / f"{s}_ablation.csv") for s in ("ppg", "ecg") if (D / f"{s}_ablation.csv").exists()}
display(Markdown(f"Model internals loaded for {', '.join(art)} (`runs/breathnet_f0/`)."))

Model internals loaded for PPG, ECG (`runs/breathnet_f0/`).

## 1 · Where the parameters are, and where the accuracy comes from

Each component of the **trained** model is switched off in turn at inference time (no retraining) and the
per-patient MAE on the held-out patients is recomputed. Left: the full ablation table. Right: the parameters
each component holds against what removing it costs, on log axes. The components that hold almost no
parameters do most of the work.

In [2]:
fig = make_subplots(1, 2, column_widths=[0.55, 0.45], horizontal_spacing=0.18,
                    subplot_titles=("Per-patient MAE with one component switched off", "Parameters held vs accuracy lost when removed"))
for sig, t in abl.items():
    t = t.set_index("variant")
    order = [v for v in t.index if v != "full model"][::-1]
    fig.add_trace(go.Bar(y=order, x=t.loc[order, "patient_mae"], orientation="h", name=sig, marker_color=SIGCOL[sig],
                         hovertemplate=sig + " · %{y}: %{x:.2f} bpm<extra></extra>"), 1, 1)
    fig.add_vline(x=np.log10(t.loc["full model", "patient_mae"]), line_color=SIGCOL[sig], line_dash="dot", line_width=2, row=1, col=1)
comp = {"bank": ("resonators at initial width", "resonator widths"), "harm": ("no harmonic summation", "harmonic weights"),
        "refine": ("no frequency refinement", "refine layer"), "frame_score": ("uniform attention over time", "frame attention"),
        "gate": ("uniform pathway gate", "reliability gate"), "rate_bias": ("no learned rate prior", "rate prior")}
for sig, a in art.items():
    params = json.loads(str(a["params"])); t = abl[sig].set_index("variant")
    xs = [params[c] for c in comp]; ys = [max(t.loc[v, "delta_patient_mae"], 0.01) for v, _ in comp.values()]
    pos = {"bank": "bottom right", "harm": "middle right", "refine": "middle left", "frame_score": "top left",
           "gate": "top center", "rate_bias": "bottom left"}
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="markers+text", text=[n for _, n in comp.values()] if sig == "PPG" else None,
                             textposition=[pos[c] for c in comp], textfont=dict(size=11, color=INK2), name=sig, showlegend=False,
                             marker=dict(size=12, color=SIGCOL[sig], line=dict(color="white", width=2)),
                             hovertemplate=sig + ": %{x} parameters → +%{y:.2f} bpm<extra></extra>"), 1, 2)
fig.update_xaxes(title_text="per-patient MAE (bpm, log scale)", type="log", row=1, col=1)
fig.update_xaxes(title_text="parameters (log scale)", type="log", range=[0.5, 4.3], row=1, col=2)
fig.update_yaxes(title_text="extra per-patient MAE when removed (bpm, log)", type="log", row=1, col=2)
fig.update_layout(barmode="group")
show(fig, 520)
display(pd.concat({s: t.set_index("variant")[["patient_mae", "delta_patient_mae", "within_2bpm"]] for s, t in abl.items()}, axis=1)
        .style.format("{:.2f}").format("{:.0%}", subset=[(s, "within_2bpm") for s in abl]))

Dotted lines mark the full model (PPG 1.27, ECG 1.44). Two removals are catastrophic: the refine layer (+11–12 bpm) and the
fusion of pathways (any single pathway alone is 1–12 bpm worse). Harmonic summation is the next most
important. Frame attention holds 91 % of the parameters but is worth only 0.1–0.25 bpm.

The refine result needs care: the layers after it were trained on refined evidence, so removing it measures
how much the model *depends* on it, not what it would contribute to a model trained without it. Section 4
shows what it computes.

## 2 · The resonator bank learned to listen briefly

Each resonator is a complex Gabor filter, a damped oscillation at one rate. Its width σ sets the trade-off:
a long filter resolves nearby rates, a short one localises *when* the breathing is clean. Training moved
almost every filter from its starting width (up to 8 s) to about **2 seconds**: time resolution won, so the
attention stage can pick clean moments. The cost is frequency resolution, which harmonic summation and
fusion recover.

In [3]:
fig = make_subplots(1, 2, column_widths=[0.42, 0.58], horizontal_spacing=0.1,
                    subplot_titles=("Learned filter width σ by rate", "Learned filters, PPG model (rows) — gain over frequency"))
for sig, a in art.items():
    fig.add_trace(go.Scatter(x=a["rates"], y=a["sigma"], name=f"{sig} learned", line=dict(color=SIGCOL[sig], width=2.5)), 1, 1)
a0 = next(iter(art.values()))
fig.add_trace(go.Scatter(x=a0["rates"], y=a0["sigma_init"], name="initial width", line=dict(color=GRAY, width=2, dash="dash")), 1, 1)
a = art.get("PPG", a0)
fig.add_trace(go.Heatmap(x=a["f_grid"], y=a["rates"], z=a["freq_response"], colorscale="Blues", showscale=False,
                         hovertemplate="filter %{y:.0f} bpm at %{x:.1f} bpm: gain %{z:.2f}<extra></extra>"), 1, 2)
fig.add_trace(go.Scatter(x=[4, 60], y=[4, 60], mode="lines", line=dict(color=ORANGE, width=1, dash="dot"), showlegend=False, hoverinfo="skip"), 1, 2)
fig.update_xaxes(title_text="filter rate (bpm)", row=1, col=1); fig.update_yaxes(title_text="σ (seconds)", row=1, col=1)
fig.update_xaxes(title_text="input frequency (bpm)", range=[0, 70], row=1, col=2); fig.update_yaxes(title_text="filter rate (bpm)", row=1, col=2)
show(fig, 460)

In [4]:
a = art.get("PPG", a0); t = a["kernel_t"]; nk = len(a["rates"])
fig = make_subplots(1, 4, shared_yaxes=True, subplot_titles=[f"{r} bpm filter" for r in (6, 15, 25, 45)])
for j, r in enumerate((6, 15, 25, 45), start=1):
    k = int(r - a["rates"][0])
    fig.add_trace(go.Scatter(x=t, y=a["kernels"][k], line=dict(color=BLUE, width=2), name="real", showlegend=j == 1), 1, j)
    fig.add_trace(go.Scatter(x=t, y=a["kernels"][nk + k], line=dict(color=AQUA, width=1.5), name="imaginary", showlegend=j == 1), 1, j)
fig.update_xaxes(title_text="seconds", range=[-10, 10])
fig.update_layout(title="What the learned resonators look like (PPG model): about one or two breaths long")
show(fig, 300)

## 3 · Harmonic summation: 9 weights, worth 1.1–1.5 bpm

Breathing waveforms are not sinusoids (inspiration is shorter than expiration), so energy also appears at
2× and 3× the rate. Each pathway learns how much of that to add back to the fundamental, and how much
evidence at *half* the rate to subtract. A negative half-rate weight means "if there is energy at k/2, rate k
may only be its harmonic", an octave-error veto. Every pathway in both models learned one.

In [5]:
fig = make_subplots(1, len(art), subplot_titles=[f"{s} model" for s in art], horizontal_spacing=0.12)
for j, (sig, a) in enumerate(art.items(), start=1):
    h = a["harm"]
    fig.add_trace(go.Heatmap(z=h, x=["+ 2× rate", "+ 3× rate", "+ ½ rate"], y=["BW", "AM", "FM"], zmid=0, zmin=-1.7, zmax=1.7,
                             colorscale=[[0, "#2a78d6"], [0.5, "#f0efec"], [1, "#e34948"]], showscale=j == len(art),
                             texttemplate="%{z:.2f}", textfont=dict(size=13),
                             hovertemplate="%{y} pathway, %{x}: weight %{z:.2f}<extra></extra>"), 1, j)
    fig.update_yaxes(autorange="reversed", row=1, col=j)
fig.update_layout(title="Learned harmonic weights (red adds evidence, blue subtracts it)")
show(fig, 320)

Which pathway gets the strong 2nd/3rd-harmonic weights differs by signal: AM in the PPG model, BW in the
ECG model. Those are the same pathways the refine layer inverts (next section): in each model one pathway is
reshaped as a whole into something other than a plain breathing detector.

## 4 · Three biased experts that cancel: the core of the model

Below, each window's evidence is z-scored and lined up on the **true** rate (offset 0), then averaged over
4,000 held-out windows. A perfect detector would show a single peak at 0.

* In PPG, **BW and FM** do peak at 0 but carry extra evidence on the *slow* side (negative offsets): that is
  the 1/f background, and alone they under-estimate by 6–7 bpm.
* **AM** is the opposite: strongly *negative* on the slow side and peaking 10 bpm too fast. It is not
  detecting breathing; it is a learned estimate of the background, inverted.
* **Fused** (gate-weighted sum, what the model decides on): the slow-side excess is gone and the peak sits on
  the true rate.

In [6]:
names = ["BW", "AM", "FM", "fused"]
fig = make_subplots(1, len(art), shared_yaxes=True, subplot_titles=[f"{s} model" for s in art], horizontal_spacing=0.05)
for j, (sig, a) in enumerate(art.items(), start=1):
    off, prof = a["aligned_offsets"], a["aligned_profiles"]
    for i, n in enumerate(names):
        c = PCOL[n]
        if n == "fused":
            fig.add_trace(go.Scatter(x=np.r_[off, off[::-1]], y=np.r_[prof[i, 2], prof[i, 1][::-1]], fill="toself",
                                     fillcolor="rgba(11,11,11,0.08)", line=dict(width=0), hoverinfo="skip", showlegend=False), 1, j)
        fig.add_trace(go.Scatter(x=off, y=prof[i, 0], name=n, legendgroup=n, showlegend=j == 1,
                                 line=dict(color=c, width=3.5 if n == "fused" else 2),
                                 hovertemplate=n + ": offset %{x} bpm, mean z %{y:.2f}<extra></extra>"), 1, j)
    fig.add_vline(x=0, line_color=GRAY, line_dash="dot", row=1, col=j)
    fig.add_vrect(x0=-20, x1=-10, fillcolor=GRAY, opacity=0.12, line_width=0, row=1, col=j)
fig.update_xaxes(title_text="rate minus true rate (bpm)")
fig.update_yaxes(title_text="evidence (z-score, mean over windows)", row=1, col=1)
fig.update_layout(title="Evidence lined up on the true rate (shaded = slow side, where the 1/f background sits)",
                  )
show(fig, 440)

In [7]:
rows = []
for sig, a in art.items():
    ref = a["sample_ref"]
    for i, n in enumerate(["BW", "AM", "FM"]):
        e = a["sample_pathway_rr"][:, i] - ref
        rows.append({"model": sig, "evidence": f"{n} alone (argmax)", "MAE": np.abs(e).mean(), "bias": e.mean(), "peaks ≤ 8 bpm": np.mean(a["sample_pathway_rr"][:, i] <= 8)})
    e = a["sample_fused_rr"] - ref
    rows.append({"model": sig, "evidence": "fused (argmax)", "MAE": np.abs(e).mean(), "bias": e.mean(), "peaks ≤ 8 bpm": np.mean(a["sample_fused_rr"] <= 8)})
display(pd.DataFrame(rows).set_index(["model", "evidence"]).style.format({"MAE": "{:.2f}", "bias": "{:+.2f}", "peaks ≤ 8 bpm": "{:.0%}"}))
fig = make_subplots(1, len(art), shared_yaxes=True, subplot_titles=[f"{s} model" for s in art])
for j, (sig, a) in enumerate(art.items(), start=1):
    for i, n in enumerate(["BW", "AM", "FM"]):
        fig.add_trace(go.Histogram(x=a["sample_pathway_rr"][:, i] - a["sample_ref"], xbins=dict(start=-30, end=30, size=1), name=n,
                                   legendgroup=n, showlegend=j == 1, marker_color=PCOL[n], opacity=0.55), 1, j)
    fig.add_trace(go.Histogram(x=a["sample_fused_rr"] - a["sample_ref"], xbins=dict(start=-30, end=30, size=1), name="fused",
                               legendgroup="fused", showlegend=j == 1, marker_color=INK, opacity=0.85), 1, j)
fig.update_layout(barmode="overlay", title="Error of each pathway alone vs fused (estimate − reference)")
fig.update_xaxes(title_text="bpm"); fig.update_yaxes(title_text="windows", row=1, col=1)
show(fig, 380)

### What the refine layer computes

The refine layer is two small convolutions along the rate axis, added back to its input. Comparing its output
with its input over the same windows shows it is not sharpening peaks. It acts as a **gain per pathway**:
it amplifies some pathways and **inverts** one, pushing it down hard at the slowest rates. That inversion is
what turns AM (PPG) or BW (ECG) into the background estimate above.

In [8]:
fig = make_subplots(1, len(art) + 1, column_widths=[0.2] + [0.4] * len(art), horizontal_spacing=0.08,
                    subplot_titles=["correlation of change with input"] + [f"{s}: mean change added by refine" for s in art])
for sig, a in art.items():
    fig.add_trace(go.Bar(x=["BW", "AM", "FM"], y=a["refine_corr"], name=sig, marker_color=SIGCOL[sig],
                         hovertemplate=sig + " %{x}: r = %{y:.2f}<extra></extra>"), 1, 1)
for j, (sig, a) in enumerate(art.items(), start=2):
    for i, n in enumerate(["BW", "AM", "FM"]):
        fig.add_trace(go.Scatter(x=a["rates"], y=a["refine_delta_mean"][i], name=n, legendgroup=n, showlegend=j == 2,
                                 line=dict(color=PCOL[n], width=2.5)), 1, j)
    fig.add_hline(y=0, line_color=GRAY, row=1, col=j)
    fig.update_xaxes(title_text="rate (bpm)", row=1, col=j)
fig.update_yaxes(range=[-1, 1], row=1, col=1)
fig.update_layout(barmode="group", title="Refine: amplify, or invert into a background estimate")
show(fig, 400)

## 5 · The gate and the attention: adaptive, but small

The reliability gate weights each pathway per window, a learned version of smart fusion. In the PPG model it
is almost a fixed weighting (narrow boxes); in the ECG model it adapts much more. Frame attention spreads over
most of the window for BW and FM but, in the PPG model, concentrates AM on about 10 of the 60 frames.

In [9]:
fig = make_subplots(1, 2, subplot_titles=("Reliability gate weight per pathway", "Frames effectively attended (of 60)"), horizontal_spacing=0.1)
for sig, a in art.items():
    g = a["sample_gate"]; att = a["sample_attention"]
    eff = np.exp(-(att * np.log(att + 1e-9)).sum(-1))
    for i, n in enumerate(["BW", "AM", "FM"]):
        fig.add_trace(go.Box(y=g[:, i], x=[n] * len(g), name=sig, legendgroup=sig, showlegend=i == 0, marker_color=SIGCOL[sig],
                             boxpoints=False, offsetgroup=sig), 1, 1)
        fig.add_trace(go.Box(y=eff[:, i], x=[n] * len(eff), name=sig, legendgroup=sig, showlegend=False, marker_color=SIGCOL[sig],
                             boxpoints=False, offsetgroup=sig), 1, 2)
fig.update_layout(boxmode="group")
fig.update_yaxes(title_text="weight", row=1, col=1); fig.update_yaxes(title_text="frames (exp. of attention entropy)", row=1, col=2)
show(fig, 400)

## 6 · Inside single windows

For each example window, two views on the same 60 s:

* **Time:** the impedance reference with its breaths (top), then each input pathway with the model's attention
  over time drawn as a grey profile behind it (taller = more weight), and below it the integrated-gradients
  attribution: which input samples pushed the predicted rate's score up (coloured) or down (grey).
* **Rate:** the energy map each pathway's resonators produced (rate × time, after harmonics and refine), the
  per-pathway evidence with its gate weight, and the final distribution against the reference.

In [10]:
def window_views(sig, j):
    a = art[sig]; meta = json.loads(str(a[f"ex{j}_meta"])); rates = a["rates"]
    x, att, ig = a[f"ex{j}_input"], a[f"ex{j}_attention"], a[f"ex{j}_ig"]
    resp, breaths = a[f"ex{j}_resp"], a[f"ex{j}_breaths"]
    t_in = np.arange(x.shape[1]) / 4.0; t_fr = np.arange(att.shape[1]) * (60 / att.shape[1])
    title = (f"{sig} · {meta['label']} — {meta['patient_id']} at {meta['start_s']:.0f} s · reference {meta['ref_rr']:.1f}, "
             f"BreathNet {meta['pred']:.1f}" + (f", CNN {meta['cnn']:.1f}" if meta['cnn'] is not None else "") + " bpm")
    display(Markdown(f"### {title}"))
    fig = make_subplots(7, 1, shared_xaxes=True, vertical_spacing=0.025, row_heights=[1, 1, 0.6, 1, 0.6, 1, 0.6],
                        subplot_titles=["impedance reference · breaths", "BW input · grey = attention", "BW attribution",
                                        "AM input · grey = attention", "AM attribution", "FM input · grey = attention", "FM attribution"])
    tr = np.arange(len(resp)) / 4.0
    fig.add_trace(go.Scatter(x=tr, y=resp, line=dict(color=INK2, width=2), showlegend=False), 1, 1)
    bi = np.clip(np.round(breaths * 4).astype(int), 0, len(resp) - 1)
    fig.add_trace(go.Scatter(x=breaths, y=resp[bi], mode="markers", marker=dict(color=INK, size=8), showlegend=False,
                             hovertemplate="breath %{x:.1f} s<extra></extra>"), 1, 1)
    for i, n in enumerate(["BW", "AM", "FM"]):
        r = 2 + 2 * i
        lo, hi = float(x[i].min()), float(x[i].max())
        w = lo + (hi - lo) * att[i] / att[i].max()            # attention, scaled to the input's range
        xs_att = np.r_[t_fr, t_fr[-1] + 1]
        fig.add_trace(go.Scatter(x=xs_att, y=np.full(len(xs_att), lo), line=dict(width=0), showlegend=False, hoverinfo="skip"), r, 1)
        fig.add_trace(go.Scatter(x=xs_att, y=np.r_[w, w[-1]], line=dict(width=0, shape="hv"), fill="tonexty",
                                 fillcolor="rgba(137,135,129,0.25)", showlegend=False, hoverinfo="skip"), r, 1)
        fig.update_yaxes(range=[lo - 0.05 * (hi - lo), hi + 0.05 * (hi - lo)], row=r, col=1)
        fig.add_trace(go.Scatter(x=t_in, y=x[i], line=dict(color=PCOL[n], width=1.8), showlegend=False,
                                 hovertemplate=n + " %{x:.1f} s<extra></extra>"), r, 1)
        fig.add_trace(go.Bar(x=t_in, y=ig[i], marker_color=np.where(ig[i] >= 0, PCOL[n], GRAY), showlegend=False,
                             hovertemplate="attribution %{y:.3f}<extra></extra>"), r + 1, 1)
        for b in breaths:
            fig.add_vline(x=b, line_color=GRAY, line_width=1, line_dash="dot", row=r, col=1)
    fig.update_yaxes(showticklabels=False); fig.update_xaxes(title_text="time in window (s)", row=7, col=1)
    fig.update_annotations(font_size=11, x=0, xanchor="left")
    show(fig, 820)

    e_map, ev, gate, prob = a[f"ex{j}_refined_energy"], a[f"ex{j}_evidence"], a[f"ex{j}_gate"], a[f"ex{j}_prob"]
    fig = make_subplots(1, 5, column_widths=[0.2, 0.2, 0.2, 0.2, 0.2], horizontal_spacing=0.035,
                        subplot_titles=["BW energy (rate × time)", "AM energy", "FM energy", "evidence per pathway", "model output"])
    for i, n in enumerate(["BW", "AM", "FM"]):
        fig.add_trace(go.Heatmap(z=e_map[i], x=t_fr, y=rates, colorscale="Blues", showscale=False,
                                 hovertemplate=n + ": %{y:.0f} bpm at %{x:.0f} s<extra></extra>"), 1, i + 1)
        fig.add_hline(y=meta["ref_rr"], line_color=ORANGE, line_width=1.5, line_dash="dot", row=1, col=i + 1)
        evn = (ev[i] - ev[i].mean()) / (ev[i].std() + 1e-9)
        fig.add_trace(go.Scatter(x=evn, y=rates, line=dict(color=PCOL[n], width=2), name=f"{n} (gate {gate[i]:.2f})"), 1, 4)
    fig.add_hline(y=meta["ref_rr"], line_color=ORANGE, line_width=1.5, line_dash="dot", row=1, col=4)
    fig.add_trace(go.Scatter(x=prob, y=rates, fill="tozerox", line=dict(color=INK, width=2), name="P(rate)",
                             fillcolor="rgba(11,11,11,0.12)"), 1, 5)
    fig.add_hline(y=meta["ref_rr"], line_color=ORANGE, line_width=2, row=1, col=5,
                  annotation_text=f"reference {meta['ref_rr']:.1f}", annotation_position="top right", annotation_font_color=ORANGE)
    fig.update_yaxes(title_text="breathing rate (bpm)", row=1, col=1)
    for c in range(2, 6): fig.update_yaxes(showticklabels=False, row=1, col=c)
    fig.update_xaxes(title_text="time (s)", row=1, col=1); fig.update_xaxes(title_text="z", row=1, col=4)
    show(fig, 470)
    # where is the impedance most disturbed, and how much attention went there?
    blocks = np.array([np.subtract(*np.percentile(resp[k * 20:(k + 1) * 20], [75, 25])) for k in range(12)])
    b0 = int(blocks.argmax()); in_b = (t_fr >= 5 * b0) & (t_fr < 5 * b0 + 5)
    share = ", ".join(f"{n} {att[i][in_b].sum():.3f}" for i, n in enumerate(["BW", "AM", "FM"]))
    display(Markdown(f"Most disturbed 5 s of the impedance: {5 * b0}–{5 * b0 + 5} s ({blocks.max() / np.median(blocks):.1f}× the median "
                     f"breath amplitude). Attention given to it: {share} (an even spread would give {in_b.mean():.3f})."))

for sig, a in art.items():
    for j in range(int(a["n_examples"])):
        window_views(sig, j)

### PPG · Typical window — 01NVA-003-2110 at 55530 s · reference 22.7, BreathNet 22.2, CNN 22.2 bpm

Most disturbed 5 s of the impedance: 50–55 s (1.1× the median breath amplitude). Attention given to it: BW 0.099, AM 0.021, FM 0.054 (an even spread would give 0.083).

### PPG · One pathway misleads, fusion recovers — 01NVA-003-2115 at 46080 s · reference 23.9, BreathNet 23.9, CNN 24.2 bpm

Most disturbed 5 s of the impedance: 5–10 s (1.4× the median breath amplitude). Attention given to it: BW 0.077, AM 0.002, FM 0.093 (an even spread would give 0.083).

### PPG · Movement burst inside the window — 01NVA-003-2223 at 46140 s · reference 25.2, BreathNet 25.3, CNN 24.9 bpm

...

Most disturbed 5 s of the impedance: 45–50 s (1.3× the median breath amplitude). Attention given to it: BW 0.095, AM 0.082, FM 0.066 (an even spread would give 0.083).

### ECG · One pathway misleads, fusion recovers — 01NVA-003-2126 at 570 s · reference 27.0, BreathNet 26.8, CNN 26.5 bpm

Most disturbed 5 s of the impedance: 30–35 s (1.1× the median breath amplitude). Attention given to it: BW 0.064, AM 0.093, FM 0.113 (an even spread would give 0.083).

### ECG · Movement burst inside the window — 01NVA-003-2223 at 46140 s · reference 25.2, BreathNet 25.8, CNN 25.4 bpm

Most disturbed 5 s of the impedance: 40–45 s (4.0× the median breath amplitude). Attention given to it: BW 0.008, AM 0.017, FM 0.097 (an even spread would give 0.083).

What to look for:

* **Typical window:** a steady horizontal band at the reference (dotted orange) in the BW and FM energy maps,
  while the inverted AM map is dark at fast rates and light at slow ones: it is the background estimate, not a
  breathing detector.
* **One pathway misleads:** individual pathways peak far from the reference (PPG: AM at 34, FM at 45 bpm), yet
  the fused distribution sits on the true rate. The product of experts in a single window.
* **Movement burst:** the line under each example measures it. In the PPG window the impedance is disturbed at
  40–45 s; BW gives that stretch 0.011 of its attention and AM none, against 0.083 for an even spread, so the
  estimate stays on the reference. The *attribution* there is still high: those samples are what drive the
  frame scorer to switch attention off, and attribution measures influence on the output, not where the
  evidence was read from.

## 7 · BreathNet and the CNN: same answers?

If the two models made different mistakes, averaging them would help. Their errors are compared per window on
the same held-out windows.

In [11]:
rows = []
for sig in art:
    s = sig.lower()
    b = pd.read_csv(f"runs/{s}_modwave_w60_breathnet_f0/test_predictions.csv").set_index("window_row")
    c = pd.read_csv(f"runs/{s}_mod_w60_f0/test_predictions.csv").set_index("window_row")
    j = c[["patient_id", "ref_rr", "pred_rr"]].join(b[["pred_rr"]], rsuffix="_bn", how="inner")
    eb, ec = j.pred_rr_bn - j.ref_rr, j.pred_rr - j.ref_rr
    ens = (j.pred_rr_bn + j.pred_rr) / 2 - j.ref_rr
    pm = lambda e: e.abs().groupby(j.patient_id).mean().mean()
    rows.append({"signal": sig, "windows": len(j), "error correlation": np.corrcoef(eb, ec)[0, 1],
                 "BreathNet per-patient MAE": pm(eb), "CNN per-patient MAE": pm(ec), "average of both": pm(ens)})
    per = pd.DataFrame({"bn": eb.abs(), "cnn": ec.abs(), "p": j.patient_id}).groupby("p").mean()
    if sig == "PPG":
        fig = go.Figure(go.Scatter(x=per.cnn, y=per.bn, mode="markers", marker=dict(size=9, color=BLUE, line=dict(color="white", width=1.5)),
                                   text=per.index, hovertemplate="%{text}<br>CNN %{x:.2f}, BreathNet %{y:.2f}<extra></extra>"))
        mx = float(max(per.max())) * 1.05
        fig.add_trace(go.Scatter(x=[0, mx], y=[0, mx], mode="lines", line=dict(color=GRAY, dash="dot"), hoverinfo="skip"))
        fig.update_layout(title="PPG: per-patient MAE, BreathNet vs CNN (points below the line: BreathNet better)",
                          xaxis_title="CNN (bpm)", yaxis_title="BreathNet (bpm)", showlegend=False)
        show(fig, 440)
display(pd.DataFrame(rows).set_index("signal").style.format({"windows": "{:,}", "error correlation": "{:.2f}",
        "BreathNet per-patient MAE": "{:.2f}", "CNN per-patient MAE": "{:.2f}", "average of both": "{:.2f}"}))

,windows,error correlation,BreathNet per-patient MAE,CNN per-patient MAE,average of both
signal,,,,,
PPG,"38,525",0.82,1.27,1.19,1.18
ECG,"52,345",0.80,1.44,1.37,1.35


The two models' errors are strongly correlated (r ≈ 0.8): they struggle on the same windows and the same
patients, the one outlier patient above included, so averaging them gains only 0.01–0.02 bpm. What limits both
is the input, not the network.

## 8 · Is it the same mechanism in every fold?

BreathNet was retrained from scratch in every cross-validation fold, on different patients. If the story
above were an accident of one training run, the folds would disagree about which parts matter and what they
learned. Each fold's model is read out the same way (`python -m scripts.breathnet_analysis` per fold).

In [12]:
folds_art = {}
for k in [int(f) for f in os.environ.get("REPORT_FOLDS", "0 1 2 3 4").split()]:
    for s in ("ppg", "ecg"):
        z, c = D.parent / f"breathnet_f{k}" / f"{s}.npz", D.parent / f"breathnet_f{k}" / f"{s}_ablation.csv"
        if z.exists() and c.exists():
            folds_art[(s.upper(), k)] = (np.load(z), pd.read_csv(c))
done = sorted({k for _, k in folds_art})
display(Markdown(f"Folds with a trained and exported BreathNet: **{done}**"
                 + ("" if len(done) == 5 else " — the rest are still training; re-run `bash scripts/build_report.sh`.")))

rows = []
for (sig, k), (a, t) in sorted(folds_art.items()):
    inv = int(np.argmin(a["refine_corr"]))
    rows.append({"signal": sig, "fold": k, "inverted pathway": ["BW", "AM", "FM"][inv], "inversion r": a["refine_corr"][inv],
                 "median resonator width (s)": float(np.median(a["sigma"][6:50])),
                 "half-rate weights (BW, AM, FM)": ", ".join(f"{w:+.2f}" for w in a["harm"][:, 2]),
                 "refine removed (+bpm)": float(t.set_index("variant").loc["no frequency refinement", "delta_patient_mae"]),
                 "harmonics removed (+bpm)": float(t.set_index("variant").loc["no harmonic summation", "delta_patient_mae"])})
learned = pd.DataFrame(rows)
display(learned.set_index(["signal", "fold"]).style.format({"inversion r": "{:+.2f}", "median resonator width (s)": "{:.1f}",
                                                            "refine removed (+bpm)": "{:+.2f}", "harmonics removed (+bpm)": "{:+.2f}"}))
for sig, g in learned.groupby("signal"):
    top = g["inverted pathway"].value_counts()
    neg = sum(all(float(w) < 0 for w in r.split(", ")) for r in g["half-rate weights (BW, AM, FM)"])
    display(Markdown(f"**{sig}**: the refine layer inverts **{top.index[0]}** in {top.iloc[0]} of {len(g)} folds"
                     + (f" (other folds: {', '.join(f'{p} ×{n}' for p, n in top.iloc[1:].items())})" if len(top) > 1 else "")
                     + f"; all three half-rate weights are negative in {neg} of {len(g)} folds; resonator widths "
                     f"{g['median resonator width (s)'].min():.1f}–{g['median resonator width (s)'].max():.1f} s."))

Folds with a trained and exported BreathNet: **[0, 1, 2, 3, 4]**

**ECG**: the refine layer inverts **BW** in 5 of 5 folds; all three half-rate weights are negative in 5 of 5 folds; resonator widths 2.1–2.2 s.

**PPG**: the refine layer inverts **AM** in 5 of 5 folds; all three half-rate weights are negative in 5 of 5 folds; resonator widths 2.0–2.2 s.

In [13]:
fig = make_subplots(1, 2, subplot_titles=[f"{s}: extra per-patient MAE when switched off" for s in ("PPG", "ECG")],
                    horizontal_spacing=0.25)
for j, sig in enumerate(("PPG", "ECG"), start=1):
    tabs = [t.assign(fold=k) for (s, k), (_, t) in folds_art.items() if s == sig]
    if not tabs: continue
    t = pd.concat(tabs); t = t[t.variant != "full model"]
    order = list(dict.fromkeys(t.variant))[::-1]
    mean = t.groupby("variant").delta_patient_mae.mean().reindex(order)
    fig.add_trace(go.Bar(y=order, x=mean.clip(lower=0.01), orientation="h", marker_color=SIGCOL[sig], opacity=0.35,
                         name=f"{sig} mean", showlegend=False, hovertemplate="%{y}: mean +%{x:.2f} bpm<extra></extra>"), 1, j)
    fig.add_trace(go.Scatter(y=t.variant, x=t.delta_patient_mae.clip(lower=0.01), mode="markers", marker=dict(color=SIGCOL[sig], size=8,
                             line=dict(color="white", width=1)), customdata=t.fold, showlegend=False,
                             hovertemplate="fold %{customdata}: +%{x:.2f} bpm<extra></extra>"), 1, j)
fig.update_xaxes(type="log", title_text="bpm (log scale); dots = folds, bars = mean")
show(fig, 480)

fig = make_subplots(1, 2, subplot_titles=("Learned resonator width per fold", "Evidence lined up on the true rate, per fold (fused)"))
dash = ["solid", "dot", "dash", "longdash", "dashdot"]
for (sig, k), (a, _) in sorted(folds_art.items()):
    fig.add_trace(go.Scatter(x=a["rates"], y=a["sigma"], line=dict(color=SIGCOL[sig], dash=dash[k], width=1.8),
                             name=f"{sig} fold {k}", legendgroup=f"{sig}{k}"), 1, 1)
    fig.add_trace(go.Scatter(x=a["aligned_offsets"], y=a["aligned_profiles"][3, 0], line=dict(color=SIGCOL[sig], dash=dash[k], width=1.8),
                             name=f"{sig} fold {k}", legendgroup=f"{sig}{k}", showlegend=False), 1, 2)
fig.update_xaxes(title_text="filter rate (bpm)", row=1, col=1); fig.update_yaxes(title_text="σ (s)", row=1, col=1)
fig.update_xaxes(title_text="rate minus true rate (bpm)", row=1, col=2); fig.update_yaxes(title_text="fused evidence (z)", row=1, col=2)
show(fig, 460)

## 9 · BreathNet-2: building the improvisations in

Sections 1–7 showed what BreathNet had to *invent*: a background estimate made by sacrificing a pathway, an
octave veto, resonator widths, and an attention stage that holds 91 % of the parameters for 0.1–0.25 bpm.
BreathNet-2 offers each of those as an explicit mechanism, switchable one at a time (`--arch breathnet2
--variant …`): **whiten** (a fitted 1/f background removed per frame, no parameters), **coherent** (a
phase-aligned sum of the three pathways' complex resonator outputs, 9 parameters), **track** (rhythm
persistence as a hidden Markov model over rate × time; replaces attention), **hr** (a heart-rate prior from the
beat intervals, 4 parameters) and **conv** (a small 2-D convolution instead of attention and gate). Every
variant is scored on the same fold-0 test windows and compared with BreathNet and the CNN by a paired
per-patient Wilcoxon test (`python -m scripts.breathnet2_summary`).

In [14]:
b2 = pd.read_csv(D.parent / "breathnet2_fold0_summary.csv") if (D.parent / "breathnet2_fold0_summary.csv").exists() else None
if b2 is None or b2["patient MAE"].notna().sum() <= 4:
    display(Markdown("BreathNet-2 runs have not been screened yet: `bash scripts/run_breathnet2_fold0.sh`."))
else:
    base = {(r["signal"], r["model"]): r["patient MAE"] for _, r in b2.iterrows() if r["model"] in ("CNN", "BreathNet")}
    v = b2[~b2.model.isin(("CNN", "BreathNet")) & b2["patient MAE"].notna()].copy()
    v["sig_bn"] = v["vs BreathNet: p"] < 0.05
    fig = make_subplots(1, 2, subplot_titles=[f"{s}: per-patient MAE on fold-0 test patients" for s in ("PPG", "ECG")], horizontal_spacing=0.08)
    for j, sig in enumerate(("PPG", "ECG"), start=1):
        t = v[v.signal == sig]
        if t.empty: continue
        col = [("#1baf7a" if (p < 0.05 and d < 0) else ("#e34948" if (p < 0.05 and d > 0) else GRAY))
               for p, d in zip(t["vs BreathNet: p"], t["vs BreathNet: d (bpm)"])]
        fig.add_trace(go.Scatter(x=t.model, y=t["patient MAE"], mode="markers", showlegend=False,
                                 marker=dict(color=col, size=16, line=dict(color="white", width=1.5)),
                             customdata=np.stack([t.params, t["vs BreathNet: better/n"], t["vs BreathNet: p"], t["vs CNN: p"]], 1),
                             hovertemplate="%{x}<br>per-patient MAE %{y:.3f}<br>%{customdata[0]:,.0f} parameters<br>"
                                           "better than BreathNet for %{customdata[1]} patients, p = %{customdata[2]:.2g}<br>"
                                           "vs CNN p = %{customdata[3]:.2g}<extra></extra>"), 1, j)
        for name, color, dash in (("BreathNet", VIOLET, "dash"), ("CNN", INK, "dot")):
            y = base[(sig, name)]
            fig.add_hline(y=y, line=dict(color=color, dash=dash, width=1.5), row=1, col=j,
                          annotation_text=f"{name} {y:.2f}", annotation_position="top left", annotation_font_color=color)
        lo = min(t["patient MAE"].min(), base[(sig, "CNN")]) - 0.06
        fig.update_yaxes(range=[lo, t["patient MAE"].max() + 0.05], row=1, col=j)
    fig.update_xaxes(tickangle=-30)
    fig.update_layout(title="Per-patient MAE (lower is better). Green = significantly better than BreathNet (paired Wilcoxon, p < 0.05); red = significantly worse")
    show(fig, 520)
    cols = ["signal", "model", "params", "MAE", "patient MAE", "within 2 bpm", "vs BreathNet: better/n", "vs BreathNet: p", "vs CNN: better/n", "vs CNN: p"]
    display(b2[cols].set_index(["signal", "model"]).style.format({"params": "{:,.0f}", "MAE": "{:.3f}", "patient MAE": "{:.3f}", "within 2 bpm": "{:.1%}",
                                                                   "vs BreathNet: p": "{:.2g}", "vs CNN: p": "{:.2g}"}, na_rep="–"))

What the mechanisms learned is physiology, not curve-fitting: the HMM step width converged to the same
~0.4 breaths/min per second on PPG and ECG independently (breathing drifts slowly); the coherent sum put BW and
FM about 40° on either side of AM (the baseline and the vagal response lead and lag the pulse amplitude);
and the heart-rate prior turned into an HR-dependent tilt of the rate prior rather than a hard HR/2
ceiling (the ceiling parameter drifted above 1 × HR, i.e. it never engages).

In [15]:
lj = D.parent / "breathnet2_fold0_learned.json"
if lj.exists():
    L = pd.DataFrame(json.loads(lj.read_text()))
    keep = [c for c in ("signal", "model", "HMM step sd (bpm/frame)", "coherent phase (deg)", "coherent lag (s)", "coherent gain BW,AM,FM",
                        "RR/HR centre", "RR/HR sd (log)", "ceiling (x HR)", "temperature", "epochs") if c in L]
    display(L[keep].set_index(["signal", "model"]).style.format(precision=2, na_rep="–"))

## Why so few parameters are enough

* **The answer is built into the structure.** A fixed-frequency resonator per rate means the network never
  has to learn frequency analysis, nor a head that maps features to rates: bin k is rate k. The CNN spends
  most of its 400k parameters learning both.
* **The physiology supplies the inductive biases.** Three pathways as separate experts, harmonics of a
  non-sinusoidal breath, steady rhythm over transient artefact. Each costs a handful of parameters and the
  ablations show each is used.
* **The hard part, rejecting the 1/f background, emerged from training.** The model turned one pathway into
  a background estimate and cancels it by summing log-evidence. This is the same problem we solved by hand
  for the signal-processing methods, solved by the network with 267 weights.
* **What the parameters are actually spent on** (frame attention, 91 %) is the least important part. A
  leaner attention could make the model smaller still.

Limits: fold 0 only; ablations measure how much the *trained* model depends on a part, not what a model
trained without it would achieve; the two examples per signal are illustrations, not evidence.